# Quickstart

This notebook runs ScisTreeCNA end to end on a small bundled dataset: load read counts
and copy numbers, infer a cell lineage tree, call genotypes, and save the results.

It needs an NVIDIA GPU. See [](../installation/gpu.md) if `import scistreecna` fails.

In [1]:
import numpy as np
import pandas as pd
import scistreecna as scna

print("ScisTreeCNA", scna.__version__)

ScisTreeCNA 1.1.0


## The input data

We begin by taking a quick look at the input data. Each row corresponds to a SNP site, and each column corresponds to a cell. In this example, the dataset contains **100 sites** and **60 cells**.

For each *(cell, site)* pair, the entry is a string in the format **`ref|alt|cn`**, where:

- **ref**: read count of the reference (wild-type) allele  
- **alt**: read count of the mutant allele  
- **cn**: observed copy number (either absolute copy number — recommended — or relative copy state)

Missing values should be encoded as:

- `.|.|cn` — if read counts are missing but copy number is available  
- `.|.|.` — if both read counts and copy number are missing
- `ref|alt|.` - if only copy number is missing

In [2]:
data = pd.read_csv('data/test_data_reads.csv', index_col=0)
data.iloc[:5, :6]

,c0,c1,c2,c3,c4,c5
s0,11|0|2,20|1|1,25|0|2,9|0|2,36|0|2,17|1|2
s1,4|0|2,23|0|1,18|0|2,14|2|2,32|0|2,5|1|2
s2,13|0|2,26|0|2,20|0|2,6|0|2,17|0|2,12|0|2
s3,15|1|3,12|0|3,15|0|2,3|0|2,20|0|3,24|0|2
s4,22|0|2,15|0|2,15|5|2,19|0|2,9|0|2,17|7|2


Next, we load the data and parse it into a NumPy `ndarray`. We also extract the `cell_names` and `site_names` from the input table.  

:::{tip}
Users may also prepare this data array directly from their own data, without explicitly creating a `.csv` file as shown above. ScisTreeCNA works directly with a `numpy.ndarray`.
:::

In [3]:
reads, cell_names, site_names = scna.util.read_csv('data/test_data_reads.csv')

print("reads shape:", reads.shape)          # (n_sites, n_cells, 3)
print("first 5 cells:", cell_names[:5])
print("first 5 sites:", site_names[:5])
print("entry (site 0, cell 0) = ref, alt, cn:", reads[0, 0])

reads shape: (100, 60, 3)
first 5 cells: ['c0', 'c1', 'c2', 'c3', 'c4']
first 5 sites: ['s0', 's1', 's2', 's3', 's4']
entry (site 0, cell 0) = ref, alt, cn: [11 0 2]


## Running inference

Before running **ScisTreeCNA**, configure the observation-model parameters, including the allelic dropout (ADO) rate, sequencing error rate, and copy-number noise. You must also specify the range of possible copy numbers using `cn_min` and `cn_max`. Note that `cn_min` must be greater than `0`, although the observed copy number in the input data may be `0`.

The `tree_batch_size` parameter controls how many candidate trees are evaluated together on the GPU. A larger value is generally faster but requires more GPU memory; a value that is too large will cause an out-of-memory error. Use `estimate_batch_sizes` to select a suitable value, as described in the [performance guide](../performance.md).

:::{tip}
If the input contains clone-averaged rather than cell-specific copy numbers, consider increasing `cn_noise` to approximately `0.5`. This prevents the model from treating smoothed copy-number estimates as precise observations.
:::

In [4]:
tree, geno = scna.infer(
    reads,
    cell_names=cell_names,
    ado=0.1,            # allelic dropout rate
    seq_error=0.01,     # sequencing error rate
    cn_noise=0.05,      # copy number noise
    cn_min=1,           # minimum copy number (must be > 0)
    cn_max=5,           # maximum copy number
    tree_batch_size=128,
    verbose=True,
)

───────────────────────────────────────── ScisTreeCNA ──────────────────────────────────────────
                                      #Cell: 60 #Site: 100                                      
                   CN_MIN: 1 CN_MAX: 5 ADO: 0.1 SEQ_ERR: 0.01 CN_NOISE: 0.05                    
                     MAX_ITER: inf TREE_BATCH_SIZE: 128 NODE_BATCH_SIZE: 64                     
───────────────────────────────────────── Local Search ─────────────────────────────────────────
[18:03:59] [Iteration 0]   Likelihood: -12454.8747                                              
           [Iteration 1]   Likelihood: -12448.8582                                              
           [Iteration 2]   Likelihood: -12443.3688                                              
           [Iteration 3]   Likelihood: -12440.0677                                              
[18:04:00] [Iteration 4]   Likelihood: -12436.9289                                              
           [Iteration 5]   Lik

## The results

`infer` returns the inferred tree and the called binary genotype matrix.

In [5]:
print(tree)

(((((((((((((((c11,c16),c1),c38),c19),((c27,c49),c37)),(c0,c53)),((c23,c57),c41)),((((c15,c35),c29),(c24,c48)),((c17,c30),c59))),((c4,c45),c10)),c50),(((((c31,c7),c36),c25),(c52,c6)),((c14,c28),c58))),((((((((((c32,c56),c5),c22),(c20,c43)),(c55,c9)),c2),((c21,c51),c18)),(c33,c46)),c47),(c13,c54))),(c12,c34)),c42),((((c26,c44),c8),c3),(c39,c40)));


The genotype matrix has one row per site and one column per cell, with `0` for wild type
and `1` for mutant. These are the *corrected* genotypes implied by the tree, not a
threshold applied to the raw read counts.

In [6]:
print("genotype matrix shape (sites x cells):", geno.shape)
print(geno[:8, :12])
print("\nfraction of mutant calls:", geno.mean().round(4))

genotype matrix shape (sites x cells): (100, 60)
[[0 0 0 0 0 0 0 0 0 0 0 0]
 [0 0 0 0 0 0 1 0 0 0 0 0]
 [0 0 0 0 0 0 0 0 0 0 0 0]
 [0 0 0 0 0 0 0 0 0 0 0 0]
 [0 0 1 0 0 1 0 0 0 1 0 0]
 [0 0 0 0 0 0 0 0 0 0 0 0]
 [1 1 0 0 0 0 0 0 0 0 0 1]
 [0 0 0 0 0 0 0 0 0 0 0 0]]

fraction of mutant calls: 0.0888


### Evaluation

Since the ground truth for this dataset is available, we can load it and evaluate performance using two metrics:  
- **Tree accuracy**, measured as $1 - Robinson\_Foulds_{norm}$
- **Genotype accuracy**


In [7]:
# load ground truth
import pickle
with open('./data/test_data_tree.pkl', 'rb') as f:
    true_tree = pickle.load(f)
true_genotype = np.loadtxt('./data/test_data_tg.txt', dtype=int)

tree_accuracy = scna.util.tree_accuracy(true_tree, tree)
geno_accuracy = scna.util.genotype_accuarcy(true_genotype, geno)
print(f'Tree accuracy: {tree_accuracy}, Genotype accuracy: {geno_accuracy}')

Tree accuracy: 0.4827586206896552, Genotype accuracy: 0.9911666666666666


We can also run **scistree2** and compare the results with those from ScisTreeCNA.


In [8]:
scistree2tree, scistree2geno = scna.external.infer_scistree2_tree(reads, cell_names=cell_names)
scistree2_tree_accuracy = scna.util.tree_accuracy(true_tree, scistree2tree)
scistree2_geno_accuracy = scna.util.genotype_accuarcy(true_genotype, scistree2geno)
print(f'Tree accuracy: {scistree2_tree_accuracy}, Genotype accuracy: {scistree2_geno_accuracy}')

Tree accuracy: 0.3793103448275862, Genotype accuracy: 0.9893333333333333


## Saving the results

The tree prints as Newick, so writing it out is just `str(tree)`.

In [9]:
with open('quickstart_tree.nwk', 'w') as f:
    f.write(str(tree))

np.savetxt('quickstart_genotype.txt', geno, fmt='%d', delimiter='\t')
print("saved")

saved


The [`scistreecna` command-line tool](../cli.md) does exactly this, so the equivalent of
this whole notebook is:

```bash
scistreecna --input data/test_data_reads.csv --output quickstart
```
